In [ ]:
import argparse
import torch
from tqdm import tqdm
import data_loader.data_loaders as module_data
import model.loss as module_loss
import model.metric as module_metric
import model.model as module_arch
from parse_config import ConfigParser
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
## next 2 lines are a hack for ipynb & interactive shells
sys.path.append("/home/jrm22n/NeSPReSO/ml_torch_templates/")
__file__ = "/home/jrm22n/NeSPReSO/ml_torch_templates/test.ipynb"

from utils import inverse_pca_transform

plt.rcParams.update({'font.size': 18})

args = argparse.ArgumentParser(description='PyTorch Template')
script_dir = os.path.dirname(os.path.abspath(__file__))
args.add_argument('-c', '--config', default=os.path.join(script_dir, 'test_config.json'), type=str,
                    help='config file path (default: None)')
args.add_argument('-r', '--resume', default=None, type=str,
                    help='path to latest checkpoint (default: None)')
args.add_argument('-d', '--device', default=None, type=str,
                    help='indices of GPUs to enable (default: all)')

config = ConfigParser.from_args(args)

logger = config.get_logger('test')

# setup data_loader instances
data_loader = getattr(module_data, config['data_loader']['type'])(
    config['data_loader']['args']['data_dir'],
    batch_size=512,
    shuffle=False,
    validation_split=0.0,
    num_workers=2
)

# build model architecture
model = config.init_obj('arch', module_arch)
logger.info(model)

# get function handles of loss and metrics
loss_fn = getattr(module_loss, config['loss'])
metric_fns = [getattr(module_metric, met) for met in config['metrics']]

logger.info('Loading checkpoint: {} ...'.format(config.config['resume']))
checkpoint = torch.load(config.config['resume'])
state_dict = checkpoint['state_dict']
if config['n_gpu'] > 1:
    model = torch.nn.DataParallel(model)
try:
    model.load_state_dict(state_dict)
except:
    # Create a new state_dict with 'module.' prefix for each key
    new_state_dict = {}
    for k, v in state_dict.items():
        new_state_dict[f'module.{k}'] = v

    # Load the new state_dict into the model
    try:
        model.load_state_dict(new_state_dict)
    except Exception as e:
        print(e)
        

# prepare model for testing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

In [ ]:
# after loading the model, we want to get results, statistics and visualizations
# 1. 4D (time, horizontal, vertical) Statistics:
#    - RMSE, MAE, bias, std, etc.
#    - average vertical statistic (per depth) # visualize_combined_results
#    - spatial distribution #plot_rmse_maps, comparisons ISOP
#    - time series histograms?

T = data_loader.T
S = data_loader.S
t_mean = data_loader.t_mean.reshape(1, -1)  # Reshape mean to have shape (1, 1801)
s_mean = data_loader.s_mean.reshape(1, -1)
t_pca = np.array(data_loader.temp_pca) # Not creating array causes error in inverse_pca_transform
s_pca = np.array(data_loader.sal_pca)
n_components = data_loader.n_components
lat = data_loader.lat
lon = data_loader.lon
aviso = data_loader.aviso
sss = data_loader.sss
sst = data_loader.sst
dates = data_loader.dates
depth = data_loader.depth
T_pcs_model = []
S_pcs_model = []
with torch.no_grad():
    for i, (data, target) in enumerate(data_loader):
        data, target = data.to(device), target.to(device)
        output = model(data).cpu().numpy()
        T_pcs_model.append(output[:, :n_components])
        S_pcs_model.append(output[:, n_components:])
        # print("data shape:", data.shape)
        # print("target shape:", target.shape)
        # print("output shape:", output.shape)

# Concatenate along the first axis
T_pcs_model = np.concatenate(T_pcs_model, axis=0)
S_pcs_model = np.concatenate(S_pcs_model, axis=0)

T_model = inverse_pca_transform(t_pca, T_pcs_model, t_mean)
S_model = inverse_pca_transform(s_pca, S_pcs_model, s_mean)

# # Printing the shapes of the variables
# print("T_model shape:", T_model.shape)
# print("S_model shape:", S_model.shape)
# print("T shape:", T.shape)
# print("S shape:", S.shape)
# print("t_mean shape:", t_mean.shape)
# print("s_mean shape:", s_mean.shape)
# print("t_pca shape:", t_pca.shape)
# print("s_pca shape:", s_pca.shape)
# print("n_components:", n_components)
# print("lat shape:", lat.shape)
# print("lon shape:", lon.shape)
# print("aviso shape:", aviso.shape)
# print("sss shape:", sss.shape)
# print("sst shape:", sst.shape)
# print("dates shape:", dates.shape)
# print("depth shape:", depth.shape)
# print("T_pcs_model shape:", T_pcs_model.shape)
# print("S_pcs_model shape:", S_pcs_model.shape)
# print("T_model shape:", T_model.shape)
# print("S_model shape:", S_model.shape)

In [ ]:
def basic_stats(true, pred, axis=0):
    # axis=0: over depths (1801)
    # axis=1, over profiles (>600)
    resid = pred - true
    rmse = np.sqrt(np.mean(resid**2, axis=axis))
    mean = np.mean(resid, axis=axis)
    mae = np.mean(np.abs(resid), axis=axis)
    std = np.std(resid, axis=axis)
    return resid, mean, rmse, mae, std
    
def visualize_stats(T_true, T_pred, S_true, S_pred, depth):
    """
    Visualize residual, rmse, mae and mean/std
    """
    T_resid, T_mean, T_rmse, T_mae, T_std = basic_stats(T_true, T_pred, axis=0)
    S_resid, S_mean, S_rmse, S_mae, S_std = basic_stats(S_true, S_pred, axis=0)
    
    fig = plt.figure(figsize=(18, 18))
    ax = fig.add_subplot(2, 2, 1)
    ax.plot(T_mean, depth, c='k', linestyle='--', linewidth=1, label='Mean')
    ax.plot(T_resid.T, depth, c='orange', linewidth=0.3, label='Residual')
    ax.set_title("Temperature Residual")
    plt.grid()
    plt.legend(["Mean", "Residual"])
    # plt.vlines(0, 0, 1800, colors='k', linestyles='--', linewidth=1)  # add zero line
    #invert axis
    ax.invert_yaxis()
    
    fig.add_subplot(2, 2, 2)
    plt.plot(T_mean, depth, c='r')
    plt.plot(T_std, depth, c='orange')
    plt.plot(T_mae, depth, c='m')
    plt.plot(T_rmse, depth, c='purple')
    plt.legend(["Mean", "Std", "MAE", "RMSE"])
    plt.title("Main statistics - Temperature")
    plt.vlines(0, 0, 1800, colors='k', linestyles='--', linewidth=1)  # add zero line
    plt.gca().invert_yaxis()
    #log x axis
    # plt.xscale('log')
    # plt.yscale('log')
    plt.grid(which='both')
    
    fig.add_subplot(2, 2, 3)
    ax.plot(S_mean, depth, c='k', linestyle='--', linewidth=1, label='Mean')
    plt.plot(S_resid.T, depth, c='c', linewidth=0.3, label='Residual')
    plt.title("Salinity Residual")
    plt.vlines(0, 0, 1800, colors='k', linestyles='--', linewidth=1)  # add zero line
    plt.gca().invert_yaxis()
    plt.legend(["Mean", "Residual"])
    plt.grid()
    
    fig.add_subplot(2, 2, 4)
    plt.plot(S_mean, depth, c='g')
    plt.plot(S_std, depth, c='c', linestyle='--')
    plt.plot(S_mae, depth, c='b')
    plt.plot(S_rmse, depth, c='y')
    plt.title("Main statistics - Salinity")
    #legend
    plt.legend(["Residual Mean", "Std", "MAE", "RMSE"])
    plt.vlines(0, 0, 1800, colors='k', linestyles='--', linewidth=1)  # add zero line
    plt.gca().invert_yaxis()
    # plt.xscale('log')
    # plt.yscale('log')
    plt.grid(which='both')
    
    plt.show()
     
visualize_stats(T, T_model, S, S_model, depth)
T_resid, T_mean, T_rmse, T_mae, T_std = basic_stats(T, T_model)
S_resid, S_mean, S_rmse, S_mae, S_std = basic_stats(S, S_model)

In [ ]:
## ISOP stuff from singleFileModel_SAT.py, TODO: fix
import xarray as xr
# load ISOP results
file_path_new = '/home/jrm22n/NeSPReSO/ml_torch_templates/data/ISOP1_rmse_bias_1deg_maps.nc'
data_ISOP = xr.open_dataset(file_path_new)

# Create bins for longitude and latitude
lon_bins = np.arange(np.min(data_ISOP.lon) - 0.5, np.max(data_ISOP.lon) + 1.5, 1)
lat_bins = np.arange(np.min(data_ISOP.lat) - 0.5, np.max(data_ISOP.lat) + 1.5, 1)

bin_size = 1 # degrees

# Calculate centers of the bins
lon_centers = lon_bins + bin_size/2
lat_centers = lat_bins + bin_size/2

# Initialize a NaN array for the number of profiles
num_prof = np.full((len(lat_centers), len(lon_centers)), np.nan)

# Extracting RMSE data and ensuring it matches the dimensions of the bins
avg_rmse_isop_t = data_ISOP['t_rmse_syn']
avg_rmse_isop_s = data_ISOP['s_rmse_syn']

avg_rmse_gdem_t = data_ISOP['t_rmse_gdem']
avg_rmse_gdem_s = data_ISOP['s_rmse_gdem']

avg_bias_isop_t = data_ISOP['t_bias_syn']
avg_bias_isop_s = data_ISOP['s_bias_syn']

ist = xr.open_dataset('/home/jrm22n/NeSPReSO/ml_torch_templates/data/isop1_stats_temp.nc')
iss = xr.open_dataset('/home/jrm22n/NeSPReSO/ml_torch_templates/data/isop1_stats_salt.nc')

isop_depths = ist.depth.values

fig = plt.figure(figsize=(18,18))
ax = fig.add_subplot(2,2,1)
ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
ax.grid(color='gray', linestyle='--', linewidth=0.5)
#     axs[0,0].fill_betweenx(depth_levels, nn_temp_rmse_depth - nn_temp_std_depth, nn_temp_rmse_depth + nn_temp_std_depth, color='xkcd:dark red', alpha=0.1, label='Avg T RMSE ± 1 std: NN')
plt.plot(ist.rmse.values, ist.depth.values, linewidth = 3, label = 'ISOP', color='xkcd:blue')
# ax.fill_betweenx(ist.depth.values, ist.rmse.values - ist.mad.values, ist.rmse.values + ist.mad.values, color='xkcd:blue', alpha=0.1, label='± mad')
# ax.fill_betweenx(our_depths, avg_gem_temp_rmse - gem_temp_rmse_mad, avg_gem_temp_rmse + gem_temp_rmse_mad, color='xkcd:orange', alpha=0.1, label='± mad')
plt.plot(T_rmse, depth, linewidth = 3, label = 'NeSPReSO', color='xkcd:gray')
# ax.fill_betweenx(our_depths, avg_nn_temp_rmse - nn_temp_rmse_mad, avg_nn_temp_rmse + nn_temp_rmse_mad, color='xkcd:gray', alpha=0.1, label='± mad')
ax.invert_yaxis()
plt.legend()
plt.xlabel("Temperature RMSE [°C]")
plt.ylabel("Depth [m]")
plt.title("Average temperature RMSE")

ax = fig.add_subplot(2,2,2)
ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
ax.grid(color='gray', linestyle='--', linewidth=0.5)
plt.plot(iss.rmse.values, iss.depth.values, linewidth = 3, label = 'ISOP', color='xkcd:green')
# ax.fill_betweenx(iss.depth.values, iss.rmse.values - iss.mad.values, iss.rmse.values + iss.mad.values, color='xkcd:green', alpha=0.1, label='± mad')
# ax.fill_betweenx(our_depths, avg_gem_sal_rmse - gem_sal_rmse_mad, avg_gem_sal_rmse + gem_sal_rmse_mad, color='xkcd:pink', alpha=0.1, label='± mad')
plt.plot(S_rmse, depth, linewidth = 3, label = 'NeSPReSO', color='xkcd:gray')
# ax.fill_betweenx(our_depths, avg_nn_sal_rmse - nn_sal_rmse_mad, avg_nn_sal_rmse + nn_sal_rmse_mad, color='xkcd:gray', alpha=0.1, label='± mad')
ax.invert_yaxis()
plt.legend()
plt.xlabel("Salinity RMSE [PSU]")
plt.title("Average salinity RMSE")

ax = fig.add_subplot(2,2,3)
ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
ax.grid(color='gray', linestyle='--', linewidth=0.5)
plt.plot(ist.bias.values, ist.depth.values, linewidth = 3, label = 'ISOP', color='xkcd:blue')
# ax.fill_betweenx(ist.depth.values, ist.bias.values - ist.mad.values, ist.bias.values + ist.mad.values, color='xkcd:blue', alpha=0.1, label='± mad')
# ax.fill_betweenx(our_depths, avg_gem_temp_bias - gem_temp_bias_mad, avg_gem_temp_bias + gem_temp_bias_mad, color='xkcd:orange', alpha=0.1, label='± mad')
plt.plot(np.mean(T_resid, axis = 0), depth, linewidth = 3, label = 'NeSPReSO', color='xkcd:gray')
# ax.fill_betweenx(our_depths, avg_nn_temp_bias - nn_temp_bias_mad, avg_nn_temp_bias + nn_temp_bias_mad, color='xkcd:gray', alpha=0.1, label='± mad')
ax.invert_yaxis()
plt.legend()
plt.xlabel("Temperature Bias [°C]")
plt.ylabel("Depth [m]")
plt.title("Average temperature Bias")

ax = fig.add_subplot(2,2,4)
ax.axvline(0, color='k', linestyle='--', linewidth=0.5)
ax.grid(color='gray', linestyle='--', linewidth=0.5)
plt.plot(iss.bias.values, iss.depth.values, linewidth = 3, label = 'ISOP', color='xkcd:green')
# ax.fill_betweenx(iss.depth.values, iss.bias.values - iss.mad.values, iss.bias.values + iss.mad.values, color='xkcd:green', alpha=0.1, label='± mad')
# ax.fill_betweenx(our_depths, avg_gem_sal_bias - gem_sal_bias_mad, avg_gem_sal_bias + gem_sal_bias_mad, color='xkcd:pink', alpha=0.1, label='± mad')
plt.plot(np.mean(S_resid, axis = 0), depth, linewidth = 3, label = 'NeSPReSO', color='xkcd:gray')
# ax.fill_betweenx(our_depths, avg_nn_sal_bias - nn_sal_bias_mad, avg_nn_sal_bias + nn_sal_bias_mad, color='xkcd:gray', alpha=0.1, label='± mad')
ax.invert_yaxis()
plt.legend()
plt.xlabel("Salinity Bias [PSU]")
plt.title("Average salinity Bias")

In [ ]:
import cartopy.crs as ccrs

def plot_rmse_maps(lon_bins, lat_bins, avg_rmse_nn, num_prof, title_prefix, variable_plotted):
    # Calculate centers of the bins
    lon_centers = (lon_bins[:-1] + lon_bins[1:]) / 2
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    vmin = 0

    # Set up color maps and limits
    if title_prefix == "Temperature":
        units = "[°C]"
        if variable_plotted == "Bias":
            cmap = "coolwarm"
            vmax = 1
            vmin = -1
        else:
            cmap = "YlOrRd"
            vmax = 2
            vmin = 0.3
    else:
        units = "[PSU]"
        if variable_plotted == "Bias":
            cmap = "PiYG_r"
            vmax = 0.2
            vmin = -0.2
        else:
            cmap = "PuBuGn"
            vmax = 0.35
            vmin = 0            

    # Create subplot grid
    fig, ax1 = plt.subplots(1, 1, figsize=(15, 15), subplot_kw={'projection': ccrs.PlateCarree()})

    # Plot the maps
    plot_rmse_on_ax(ax1, lon_centers, lat_centers, avg_rmse_nn, num_prof, f"NeSPReSO Average {variable_plotted} - {title_prefix}")

    pcm = ax1.pcolormesh(lon_centers, lat_centers, avg_rmse_nn, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(pcm, ax=ax1, orientation="vertical", pad=0.04, fraction=0.465*(1/15), label=f"Average {variable_plotted} {units}")
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    # Set x and y ticks, 
    ax1.set_xticks(np.arange(-99, -81, 1))
    ax1.set_yticks(np.arange(18, 30, 1))
    # Add grid
    ax1.grid(color='gray', linestyle='--', linewidth=0.5)
    plt.show()

def plot_rmse_on_ax(ax, lon_centers, lat_centers, avg_rmse_grid, num_prof, title):
    ax.set_extent([-99, -81, 18, 30])  # Set to your area of interest
    ax.coastlines()

    pcm = ax.pcolormesh(lon_centers, lat_centers, avg_rmse_grid, cmap='coolwarm', vmin=-3, vmax=3)
    ax.set_title(title, fontsize=18)

    # Annotate each cell with the average RMSE value
    for i, lon in enumerate(lon_centers):
        for j, lat in enumerate(lat_centers):
            value = avg_rmse_grid[j, i]
            number = num_prof[j, i]
            if not number==0:  # Check if the value is not NaN, and if there are more than 2 profiles in the bin
                ax.text(lon, lat + 0.2, f'{number:.0f}', color='gray', ha='center', va='center', fontsize=12, transform=ccrs.PlateCarree())
                ax.text(lon, lat - 0.2, f'{value:.2f}', color='black', ha='center', va='center', fontsize=12, transform=ccrs.PlateCarree())
            
def calculate_average_rmse_per_bin(lon_bins, lat_bins, lon_val, lat_val, residuals, dpt_range=np.arange(0, 1801)):
    avg_rmse_grid = np.zeros((len(lat_bins) - 1, len(lon_bins) - 1))
    num_prof_grid = np.zeros((len(lat_bins) - 1, len(lon_bins) - 1))

    for i in range(len(lon_bins) - 1):
        for j in range(len(lat_bins) - 1):
            # Find points that fall into the current bin
            in_bin = (lon_val >= lon_bins[i]) & (lon_val < lon_bins[i + 1]) & (lat_val >= lat_bins[j]) & (lat_val < lat_bins[j + 1])
            # print(f"Bin {i}, {j}: {np.sum(in_bin)} profiles")
            if np.sum(in_bin) == 0:
                # print(f"Warning: No profiles found in bin ({i}, {j}).")
                continue
            else:
                valid_resid = residuals[in_bin, :][:, dpt_range]
                rmses = np.sqrt(np.mean(valid_resid**2, axis=0))
                avg_rmse_grid[j, i] = np.mean(rmses)
                num_prof_grid[j, i] = np.sum(in_bin)

    return avg_rmse_grid, num_prof_grid

def calculate_average_bias_per_bin(lon_bins, lat_bins, lon_val, lat_val, residuals, dpt_range=np.arange(0, 1801)):
    avg_bias_grid = np.zeros((len(lat_bins) - 1, len(lon_bins) - 1))
    num_prof_grid = np.zeros((len(lat_bins) - 1, len(lon_bins) - 1))

    for i in range(len(lon_bins) - 1):
        for j in range(len(lat_bins) - 1):
            # Find points that fall into the current bin
            in_bin = (lon_val >= lon_bins[i]) & (lon_val < lon_bins[i + 1]) & (lat_val >= lat_bins[j]) & (lat_val < lat_bins[j + 1])
            # print(f"Bin {i}, {j}: {np.sum(in_bin)} profiles")
            if np.sum(in_bin) == 0:
                # print(f"Warning: No profiles found in bin ({i}, {j}).")
                continue
            else:
                valid_resid = residuals[in_bin, :][:, dpt_range]
                biases = np.mean(valid_resid, axis=0)
                avg_bias_grid[j, i] = np.mean(biases)
                num_prof_grid[j, i] = np.sum(in_bin)

    return avg_bias_grid, num_prof_grid

# Calculate centers of the bins
lon_centers = lon_bins + bin_size/2
lat_centers = lat_bins + bin_size/2

lon_val = lon  
lat_val = lat  
pred_T_resid = T_resid
pred_S_resid = S_resid

dpt_range = isop_depths[isop_depths <= 1800].astype(int)

# Calculate average temperature RMSE
avg_temp_rmse_nn, num_prof_nn = calculate_average_rmse_per_bin(lon_bins, lat_bins, lon_val, lat_val, pred_T_resid, dpt_range)
plot_rmse_maps(lon_bins, lat_bins, avg_temp_rmse_nn, num_prof_nn, "Temperature", "RMSE")

# Calculate average salinity RMSE
avg_sal_rmse_nn, num_prof_nn = calculate_average_rmse_per_bin(lon_bins, lat_bins, lon_val, lat_val, pred_S_resid, dpt_range)
plot_rmse_maps(lon_bins, lat_bins, avg_sal_rmse_nn, num_prof_nn, "Salinity", "RMSE")

# Calculate average temperature bias
avg_nn_t_bias, num_prof_nn = calculate_average_bias_per_bin(lon_bins, lat_bins, lon_val, lat_val, pred_T_resid, dpt_range)
plot_rmse_maps(lon_bins, lat_bins, avg_nn_t_bias, num_prof_nn, "Temperature", "Bias")

# Calculate average salinity bias
avg_nn_s_bias, num_prof_nn = calculate_average_bias_per_bin(lon_bins, lat_bins, lon_val, lat_val, pred_S_resid, dpt_range)
plot_rmse_maps(lon_bins, lat_bins, avg_nn_s_bias, num_prof_nn, "Salinity", "Bias")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_profile(profile_number, T, S, T_model, S_model, t_mean, s_mean, t_pca, s_pca, depth):
    # Ensure mean vectors are broadcastable
    t_mean = t_mean.reshape(-1)  # Reshape mean to 1D array if needed
    s_mean = s_mean.reshape(-1)

    # Get the actual profiles
    T_actual = T[profile_number]
    S_actual = S[profile_number]

    # Reconstruct the PCA profiles for comparison
    T_pcs, S_pcs = data_loader.dataset.get_pcs(profile_number)
    T_reconstructed = inverse_pca_transform(t_pca, T_pcs, t_mean)
    S_reconstructed = inverse_pca_transform(s_pca, S_pcs, s_mean)

    # Get the estimated profiles from the ML model
    T_estimated = T_model[profile_number]
    S_estimated = S_model[profile_number]

    # Plot the profiles
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 8))

    # Plot Temperature Profile
    axes[0].plot(T_actual, depth, label='Actual Temperature', color='blue')
    axes[0].plot(T_reconstructed, depth, label='Reconstructed PCA Temperature', color='orange')
    axes[0].plot(T_estimated, depth, label='Estimated Temperature', color='green')
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Temperature (°C)')
    axes[0].set_ylabel('Depth (m)')
    axes[0].set_title(f'Temperature Profile - Profile Number {profile_number}')
    axes[0].legend()
    axes[0].grid(True)

    # Plot Salinity Profile
    axes[1].plot(S_actual, depth, label='Actual Salinity', color='blue')
    axes[1].plot(S_reconstructed, depth, label='Reconstructed PCA Salinity', color='orange')
    axes[1].plot(S_estimated, depth, label='Estimated Salinity', color='green')
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Salinity (PSU)')
    axes[1].set_title(f'Salinity Profile - Profile Number {profile_number}')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

# Example usage
profile_number =   # Replace with the desired profile number
plot_profile(profile_number, T, S, T_model, S_model, t_mean, s_mean, t_pca, s_pca, depth)


In [ ]:

# T_resid, T_mean, T_rmse, T_mae, T_std = basic_stats(T, T_model)
# T_resid.shape
# find the profile(s) with the highest RMSE
T_avg_rmse = np.sqrt(np.mean(T_resid**2, axis=1))
T_avg_rmse.shape
# def plot_profile(profile_number, T, S, T_model, S_model, t_mean, s_mean, t_pca, s_pca, depth):
